In [ ]:
# Task 1 - Part A

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

discount = np.array([2, 5, 8, 10, 15, 18, 22, 25, 30, 35], dtype=float)

sales = np.array([105, 130, 158, 172, 210, 235, 268, 290, 325, 355], dtype=float)

print("Discount:", discount)
print("Sales:", sales)

In [ ]:
# Task 1 - Part A

w = 0.0
b = 0.0

learning_rate = 0.0001

iterations = 100000

n = len(discount)

loss_history = []

for i in range(iterations):
    predictions = w * discount + b
    errors = predictions - sales
    loss = np.mean(errors ** 2)
    loss_history.append(loss)
    dw = (2 / n) * np.sum(discount * errors)
    db = (2 / n) * np.sum(errors)
    w = w - learning_rate * dw
    b = b - learning_rate * db


print("Final w:", w)
print("Final b:", b)
print("Starting Loss:", loss_history[0])
print("Ending Loss:", loss_history[-1])

In [ ]:
# Task 1 - Part B

plt.figure(figsize=(8, 5))

plt.plot(loss_history)

plt.xlabel("Iterations")
plt.ylabel("Mean Squared Error")
plt.title("Loss Curve - Gradient Descent")

plt.grid(True)
plt.show()

# Task 1 - Part B
#
# The loss decreases as the number of iterations increases,
# which means that gradient descent is successfully learning
# better values for w and b.
#
# If the learning rate is far too high:
# The model may overshoot the minimum of the loss function.
# This can cause the loss to oscillate or even become extremely large,
# meaning that the algorithm may fail to converge.
#
# If the learning rate is far too low:
# The model will still move toward the minimum, but the updates
# will be very small, so convergence will be very slow and
# many more iterations will be required.

In [ ]:
# Task 1 - Part C

X = discount.reshape(-1, 1)

sklearn_model = LinearRegression()
sklearn_model.fit(X, sales)

sklearn_w = sklearn_model.coef_[0]
sklearn_b = sklearn_model.intercept_

print("From Scratch:")
print("w =", w)
print("b =", b)

print("\nSklearn:")
print("w =", sklearn_w)
print("b =", sklearn_b)

print("\nDifferences:")
print("Difference in w =", abs(w - sklearn_w))
print("Difference in b =", abs(b - sklearn_b))

In [ ]:
# Task 2 - Part A

from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

degrees = [1, 2, 4, 9]

results = []

In [ ]:
# Task 2 - Part A and B

for degree in degrees:

    poly = PolynomialFeatures(degree=degree, include_bias=False)

    X_poly = poly.fit_transform(X)

    model = LinearRegression()
    model.fit(X_poly, sales)

    predictions = model.predict(X_poly)
    mae = mean_absolute_error(sales, predictions)
    mse = mean_squared_error(sales, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(sales, predictions)

    results.append({
        "Degree": degree,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

In [ ]:
# Task 2 - Part B
import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

# Task 2 - Part C
#
# Based only on the training metrics, degree 9 looks like the best
# model because it has the lowest MAE, MSE, and RMSE and the highest R².
#
# However, this conclusion can be misleading because all metrics
# were calculated on the same data used to train the models.
#
# Degree 1 may slightly underfit the relationship because it assumes
# a simple straight-line relationship.
#
# As the polynomial degree increases, the model becomes more flexible.
# This can reduce training error, but at high degrees such as degree 9,
# the model can start fitting noise and individual observations instead
# of learning the general relationship. This is called overfitting.
#
# Therefore, a very low degree can cause underfitting, while a very
# high degree can cause overfitting. We need test data to decide
# which model generalizes best.

In [ ]:
# Task 3 - Part A

X_train = discount[:-3].reshape(-1, 1)
y_train = sales[:-3]
X_test = discount[-3:].reshape(-1, 1)
y_test = sales[-3:]

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining discounts:", X_train.ravel())
print("Test discounts:", X_test.ravel())

In [ ]:
# Task 3 - Part B
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

alpha_values = [0.1, 1, 10]

ridge_results = []

for alpha in alpha_values:
    model = Pipeline([
        ("polynomial", PolynomialFeatures(degree=9, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    model.fit(X_train, y_train)


    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    train_mae = mean_absolute_error(y_train, train_predictions)
    train_mse = mean_squared_error(y_train, train_predictions)
    train_rmse = np.sqrt(train_mse)
    train_r2 = r2_score(y_train, train_predictions)

    test_mae = mean_absolute_error(y_test, test_predictions)
    test_mse = mean_squared_error(y_test, test_predictions)
    test_rmse = np.sqrt(test_mse)
    test_r2 = r2_score(y_test, test_predictions)

    ridge_results.append({
        "Alpha": alpha,
        "Train MAE": train_mae,
        "Train MSE": train_mse,
        "Train RMSE": train_rmse,
        "Train R2": train_r2,
        "Test MAE": test_mae,
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "Test R2": test_r2
    })

In [ ]:
# Task 3 - Part B

ridge_results_df = pd.DataFrame(ridge_results)

print(ridge_results_df.to_string(index=False))

# Task 3 - Part C
#
# Increasing alpha increases the strength of Ridge regularization,
# which pushes the model coefficients toward smaller values.
#
# As alpha increases, the model becomes less flexible, so its training
# performance gets worse because it is being penalized for large
# coefficients.
#
# Regularization is intended to reduce variance and prevent overfitting,
# but if alpha becomes too large, the model can have too much bias
# and start underfitting the data.
#
# In this experiment, alpha = 0.1 gives the best test performance
# among the three degree-9 Ridge models, although the test performance
# is still poor compared with a simpler model.

In [ ]:
# Task 3 - Part D

model_comparison = []

for degree in [1, 2, 4]:

    poly = PolynomialFeatures(degree=degree, include_bias=False)

    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    predictions = model.predict(X_test_poly)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    model_comparison.append({
        "Degree": degree,
        "Test MAE": mae,
        "Test MSE": mse,
        "Test RMSE": rmse,
        "Test R2": r2
    })

comparison_df = pd.DataFrame(model_comparison)

print(comparison_df.to_string(index=False))

In [ ]:
# Optional visualization

plt.figure(figsize=(10, 6))

plt.scatter(discount, sales, label="Actual Data")

x_plot = np.linspace(discount.min(), discount.max(), 200).reshape(-1, 1)

for degree in [1, 2, 4, 9]:

    poly = PolynomialFeatures(degree=degree, include_bias=False)

    X_poly = poly.fit_transform(X)
    x_plot_poly = poly.transform(x_plot)

    model = LinearRegression()
    model.fit(X_poly, sales)

    y_plot = model.predict(x_plot_poly)

    plt.plot(x_plot, y_plot, label=f"Degree {degree}")

plt.xlabel("Discount (%)")
plt.ylabel("Weekly Sales")
plt.title("Polynomial Regression Models")
plt.legend()
plt.grid(True)

plt.show()